# VoiceForge — speech backend (Colab **or** Kaggle)

Runs Chatterbox Multilingual on a free T4 and **registers itself** with your VoiceForge app. You never copy a tunnel URL again — press Run All and the backend appears as *Online* in Admin → Backends.

**Before running**
- **Colab:** Runtime → Change runtime type → **T4 GPU**
- **Kaggle:** Session options → Accelerator → **GPU T4 x2**, and **Internet → On**

Then fill in Cell 0 and **Run All**, top to bottom.

> Cell 1 prints a wall of red `pip` dependency-conflict lines about jax, opencv, cupy and friends. **Expected and harmless** — that's the platform's preinstalled data-science stack, which this backend never imports.

In [ ]:
# Cell 0 — your app details  (copy these from Admin → Backends)

VOICEFORGE_APP_URL = ""   # e.g. https://audio.bistab.com.np
REGISTRATION_TOKEN = ""   # the REGISTRATION_TOKEN you set in Vercel
BACKEND_SECRET     = ""   # the BACKEND_SECRET you set in Vercel

# Kaggle users: instead of pasting secrets here, you can put them in
# Add-ons → Secrets and leave the three lines above empty.
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    VOICEFORGE_APP_URL = VOICEFORGE_APP_URL or _s.get_secret("VOICEFORGE_APP_URL")
    REGISTRATION_TOKEN = REGISTRATION_TOKEN or _s.get_secret("REGISTRATION_TOKEN")
    BACKEND_SECRET     = BACKEND_SECRET     or _s.get_secret("BACKEND_SECRET")
    print("Loaded settings from Kaggle Secrets.")
except Exception:
    pass

import os
PROVIDER = "kaggle" if os.path.exists("/kaggle") else "colab"
os.environ["BACKEND_SECRET"] = BACKEND_SECRET
os.environ["VOICEFORGE_VOICES_DIR"] = "/tmp/voiceforge/voices"

assert VOICEFORGE_APP_URL, "Set VOICEFORGE_APP_URL above (Admin -> Backends shows it)."
assert REGISTRATION_TOKEN, "Set REGISTRATION_TOKEN above."
print(f"Provider: {PROVIDER}  ->  {VOICEFORGE_APP_URL}")

In [ ]:
# Cell 1 — Install
# The numpy pin depends on the Python version, and it is load-bearing.
# On Python 3.13 (what Colab runs now) three things must hold at once:
#   chatterbox-tts  numpy >= 2.0
#   numba (librosa) numpy <  2.3   -> "Numba needs NumPy 2.2 or less"
#   scipy           built against numpy 2.x, so numpy 1.x breaks it with
#                   "All ufuncs must have type numpy.ufunc"
# 2.2.6 is the only version that satisfies all three. On 3.12, chatterbox
# wants numpy < 2 instead.
import sys
PY313 = sys.version_info >= (3, 13)
NUMPY_PIN = "numpy==2.2.6" if PY313 else "numpy==1.26.4"
print(f"Python {sys.version_info.major}.{sys.version_info.minor} -> {NUMPY_PIN}")

!pip install -q chatterbox-tts fastapi uvicorn nest-asyncio python-multipart soundfile
!pip install -q torchvision==0.21.0
# transformers/safetensors stay exact — chatterbox breaks on newer ones.
!pip install -q --force-reinstall {NUMPY_PIN} transformers==5.2.0 safetensors==0.5.3

# cloudflared gives us a public HTTPS URL for this session.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# The server itself lives in the repo (backend/voiceforge_server.py) so Colab,
# Kaggle and Modal all run identical code. Download the current version.
!mkdir -p /content/backend && touch /content/backend/__init__.py
!wget -q -O /content/backend/voiceforge_server.py https://raw.githubusercontent.com/waytobijay/Bbox-Audio/main/backend/voiceforge_server.py

import os
_path = "/content/backend/voiceforge_server.py"
_size = os.path.getsize(_path) if os.path.exists(_path) else 0
print("\nshared server downloaded: %.1f KB" % (_size / 1000))
assert _size > 4000, "Download failed — check the repo is public, or upload backend/voiceforge_server.py by hand."
print("Installed. Now: Runtime > Restart session, then run Cell 0 and Cell 2.")

In [ ]:
# Cell 2 — Load the model
# Run this AFTER Runtime > Restart session, so the kernel picks up what Cell 1
# installed instead of the numpy it imported at startup.
import sys, torch
sys.path.insert(0, "/content")

assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. "
    "Kaggle: Session options > Accelerator > GPU.")

import numpy as np
print(torch.cuda.get_device_name(0), "| torch", torch.__version__,
      "| numpy", np.__version__,
      "| py", f"{sys.version_info.major}.{sys.version_info.minor}")

# Check the numpy/Python PAIR rather than one fixed version — see Cell 1 for
# why 3.13 needs 2.2.x and 3.12 needs 1.26.x.
_hint = "re-run Cell 1, then Runtime > Restart session, then Cell 0 and Cell 2."
if sys.version_info >= (3, 13):
    assert np.__version__.startswith("2.2"), f"numpy {np.__version__}, need 2.2.x — {_hint}"
else:
    assert np.__version__.startswith("1."), f"numpy {np.__version__}, need 1.x — {_hint}"

# These two are where a mismatched numpy actually bites, deep inside the model
# load. Import them here so the failure is readable and instant.
import scipy.special
import numba

from backend.voiceforge_server import load_models, create_app, gpu_name, VERSION
MODEL_NAMES = load_models()
print("models ready:", MODEL_NAMES)

In [ ]:
# Cell 3 — Serve, open a tunnel, and REGISTER with your app
import json, re, subprocess, threading, time, urllib.request
import nest_asyncio, uvicorn

app = create_app(provider=PROVIDER)

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error"),
    daemon=True,
).start()
time.sleep(3)

# --- public URL ---
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = None
for line in proc.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break
assert PUBLIC_URL, "Tunnel failed to start — re-run this cell."
print("tunnel:", PUBLIC_URL)

# --- register + heartbeat ---
REGISTER_URL = VOICEFORGE_APP_URL.rstrip("/") + "/api/backends/register"

def register():
    payload = json.dumps({
        "provider": PROVIDER,
        "url": PUBLIC_URL,
        "gpu": gpu_name(),
        "models": MODEL_NAMES,
        "version": VERSION,
    }).encode()
    req = urllib.request.Request(
        REGISTER_URL, data=payload, method="POST",
        headers={"Content-Type": "application/json",
                 "X-Registration-Token": REGISTRATION_TOKEN},
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())

try:
    result = register()
    print("\n" + "=" * 62)
    print("  REGISTERED — nothing to copy. Check Admin -> Backends.")
    print(f"  provider: {PROVIDER}   heartbeat: every {result.get('heartbeat_seconds', 60)}s")
    print("=" * 62 + "\n")
except Exception as e:
    print("\n*** REGISTRATION FAILED:", e)
    print("*** Check VOICEFORGE_APP_URL and REGISTRATION_TOKEN in Cell 0.")
    print("*** The backend is still running — you can add this URL by hand")
    print("*** under Admin -> Backends -> Custom:")
    print("   ", PUBLIC_URL)

# Heartbeats keep it showing Online; 3 missed beats marks it Offline.
def heartbeat_loop():
    while True:
        time.sleep(60)
        try:
            register()
        except Exception as e:
            print("heartbeat failed:", e, flush=True)

threading.Thread(target=heartbeat_loop, daemon=True).start()
print("Heartbeat running. Keep this tab open while you generate.")

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab/Kaggle disconnect idle tabs. Run this during long jobs.
import time
while True:
    time.sleep(60)
    print(".", end="", flush=True)